In [ ]:
# LOAD & INFERENSI MODEL YANG SUDAH DITRAIN
# ============================================
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import os
from PIL import Image
import xml.etree.ElementTree as ET
import glob

In [ ]:
# LOAD MODEL
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2  # Ganti sesuai jumlah class kamu (+1 untuk background)

model = fasterrcnn_resnet50_fpn(pretrained=False)
in_features = model.roi_heads.box_predictor.cls_score.in_features
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.load_state_dict(torch.load("fasterrcnn_pascalvoc.pth", map_location=device))
model.to(device)
model.eval()

# PARSE XML (untuk ambil val data)
# -----------------------------
def parse_voc_annotation(xml_folder):
    data = []
    for xml_file in glob.glob(os.path.join(xml_folder, "*.xml")):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        filename = root.find("filename").text
        data.append(filename)
    return list(set(data))

val_images = parse_voc_annotation("dataset/annotations")
img_dir = "dataset_bbox/test"

# -----------------------------
# INFERENSI & VISUALISASI
# -----------------------------
for img_file in val_images[:5]:  # ambil 5 gambar untuk contoh
    path = os.path.join(img_dir, img_file)
    image = Image.open(path).convert("RGB")
    tensor = F.to_tensor(image).unsqueeze(0).to(device)

    with torch.no_grad():
        preds = model(tensor)[0]

    img_np = tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img_np)

    boxes = preds['boxes'].cpu().numpy()
    labels = preds['labels'].cpu().numpy()
    scores = preds['scores'].cpu().numpy()

    for box, label, score in zip(boxes, labels, scores):
        if score > 0.5:
            xmin, ymin, xmax, ymax = box
            ax.add_patch(patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                           linewidth=2, edgecolor='lime', facecolor='none'))
            ax.text(xmin, ymin, f"Label {label}: {score:.2f}", color='white',
                    backgroundcolor='green', fontsize=8)

    plt.axis('off')
    plt.title(f"Inference Result: {img_file}")
    plt.show()


c:\Users\acer\AppData\Local\Programs\PythonCodingPack\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\acer\AppData\Local\Programs\PythonCodingPack\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-1-e68e2f8976cf>:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `we

FileNotFoundError: [Errno 2] No such file or directory: 'fasterrcnn_model.pth'